In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from IPython.display import Image, display  # noqa: E402

from scripts.notebook_helpers import (  # noqa: E402
    WAVELET_FREQ_MAX,
    WAVELET_FREQ_MIN,
    WAVELET_N_FREQS,
    load_joined_condition_wavelets,
    resolve_notebook_wavelet_cache_dir,
    resolve_wavelet_dir,
)
from src.analysis import assr_trials as at  # noqa: E402
from src.analysis import iva_quality  # noqa: E402
from src.analysis.iva_condition_comparison import (  # noqa: E402
    apply_component_signs,
    decompose_channel_iva,
    slice_to_band,
)
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    REAL_CONDITIONS,
    ConditionVariants,
    CoordinateSystems,
    ExclusionCategories,
    ExperimentNames,
    MusicTypeVariants,
    SpectrumTypeVariants,
)
from src.definitions.frequency import band_token, resolve_band_range  # noqa: E402
from src.io.loading import assr_electrode_mask  # noqa: E402
from src.visualization.iva_condition_plots import (  # noqa: E402
    plot_condition_mean_tf_maps,
    plot_condition_mean_topomaps,
    plot_participant_condition_tf_maps,
    plot_participant_condition_topomaps,
)
from src.visualization.iva_quality_plots import topo_info_subset  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
%matplotlib inline
print("Setup complete.")

# Band-Restricted IVA Decomposition — the 40 Hz Neighbourhood, Both Conditions Pooled

The interactive counterpart of
[`jobs/metacentrum/06-iva-condition-comparison/run_condition_comparison_gamma.pbs`](../../jobs/metacentrum/06-iva-condition-comparison/run_condition_comparison_gamma.pbs),
which runs
[`scripts/run_iva_condition_comparison.py`](../../scripts/run_iva_condition_comparison.py)
with `--band_range 30 50`. Same pipeline, same functions, a handful of participants and
a few seconds of recording — so the band, the component count and the sign anchor can be
**tuned here before a four-hour job is queued on the whole cohort**.

**What the band changes, and it is exactly one thing.**
[`decompose_channel_iva`](../../src/analysis/iva_condition_comparison.py) flattens
frequency × time into one sample axis, so on the 1 Hz-spaced 1–50 Hz grid the 40 Hz row
is **1 bin of 50**: ~98% of both the channel PCA's variance ordering and `iva_g`'s
dependence cost is computed from frequencies the ASSR does not live in, and the
components are free to spend every `--n_pca` direction on whatever is most reproducible
across recordings (drifts, alpha) without the objective penalising it.
[`slice_to_band`](../../src/analysis/iva_condition_comparison.py) cuts the frequency axis
to 30–50 Hz **before** the PCA, so every sample is a gamma-band sample.

It does **not** make IVA supervised: `iva_g` still maximises cross-recording dependence,
not stimulus locking. It removes the frequencies the objective was wasting itself on —
no more.

**Why `--band_range 30 50` and not `--band gamma`.** The two select the same 21 bins on
this grid (gamma is 30–70 Hz and the grid stops at 50), but the explicit window says so
without the reader having to know it, and it is the flag to edit for a narrower try such
as `--band_range 35 45`. The band token `30-50hz` goes into the store filename and the
plot path, so a band run sits **beside** the broadband one rather than replacing it —
which is what makes the comparison in Step 5 possible at all.

**Scope.** Load → slice → decompose → anchor the sign → draw the Placebo/Psilocybin
comparison, band-restricted and broadband side by side. It runs no statistics; the
paired tests live in
[`scripts/run_assr_snr_grid.py`](../../scripts/run_assr_snr_grid.py) and
[`scripts/run_assr_trial_stats.py`](../../scripts/run_assr_trial_stats.py), which read
the store this workflow writes.

**The one invariant this notebook exists to check** (Step 4): the fixed
**ASSR-electrode reference** at 40 Hz is the same array in a band run and in a broadband
one. It is a binary electrode mask applied to the wavelet cache — no decomposition
enters it — so it cannot depend on what the IVA was handed. If it moves, an index was
resolved on one frequency grid and used on the other, and **every** row of the analysis
is reading the wrong part of the spectrum, not just the reference.

## Configuration

In [ ]:
# ── Experiment configuration ───────────────────────────────────
# Choose the experiment: ExperimentNames.PSILO_MUSIC or ExperimentNames.ASSR
EXPERIMENT_NAME = ExperimentNames.ASSR
# The pooled dataset IS the condition here — the two real conditions below are
# stacked on the subject axis and the product is labelled Joined_<MusicType>.
CONDITION = ConditionVariants.JOINED
# Subject-axis block order: Placebo subjects first, then Psilocybin.
CONDITIONS_TO_POOL = list(REAL_CONDITIONS)
if EXPERIMENT_NAME == ExperimentNames.ASSR:
    # ASSR has no music dimension; uses a single placeholder "music type".
    MUSIC_TYPE = MusicTypeVariants.ASSR
else:
    MUSIC_TYPE = MusicTypeVariants.CLASSICAL
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# ── The band restriction — the one setting this notebook is about ──────────
# Given as (LOW, HIGH) in Hz, exactly what the job passes as --band_range. Edit this
# one line to try another window (e.g. (35.0, 45.0) for the ASSR's immediate
# neighbourhood); it is applied BEFORE the PCA, so it changes what is decomposed and
# not merely what is plotted.
#
# A standard band name works too — BAND = "gamma" is accepted by every function below
# and by --band — but on this grid gamma means 30-50 Hz anyway, and the explicit window
# says so without the reader having to know where the grid stops.
BAND: tuple[float, float] | str | None = (30.0, 50.0)
# Canonical name of the restriction, used by the store, the plot directory and every
# figure filename. A named band keeps its name; an explicit window becomes '<low>-<high>hz'.
BAND_NAME = band_token(BAND)
# The stimulation frequency the whole ASSR analysis is about. Everything downstream
# reads THIS in Hz, never a bin index carried across grids.
ASSR_FREQ = iva_quality.ASSR_FREQ  # 40.0 Hz

# ── Wavelet settings ─────────────────────────────────────────
REPRESENTATION = "power"
# Wavelet frequency grid: canonical 1 Hz-spaced grid from src.definitions.frequency
# (WAVELET_FREQ_MIN/MAX/N_FREQS), re-exported via scripts.notebook_helpers. It must
# match the grid the cache was written with, or the cache is missed and recomputed.
# The BAND is sliced out of this grid below — there is no separate per-band cache.
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

# ── Reuse / compute ──────────────────────────────────────────
# True is the intended setting: this notebook is built around reusing the two
# per-condition caches. False would recompute both from RAW_AFTER_ICA.
REUSE_WAVELETS = True

# ── Cohort, channel and time subset ────────────────────────────
# N_PAIRS_SUBSET counts *participants*, not subjects: each kept participant brings
# one recording per condition, so the pooled subject axis is 2 × N_PAIRS_SUBSET.
# Never slice the pooled subject axis by position — the leading rows are one whole
# condition block. Use ``pooled.select_participants`` (done in the loading cell).
N_PAIRS_SUBSET: int | None = 5
# The band run's sample axis is ~40% of the broadband one (21 bins of 50), but the
# LOADING peak is unchanged: the broadband cache is read in full and sliced after. So
# these subsets govern what fits in memory here exactly as they do in a broadband
# notebook — the band buys nothing at load time.
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 3000  # first N time samples

# ── Comparison figures ────────────────────────────────────────
SAVE_PLOTS = True
# Which components the comparison figures cover. None = every component.
COMPONENTS_TO_PLOT: list[int] | None = None
# Reference lines on the TF panels. The ASSR is continuous 40 Hz stimulation, so the
# stimulation frequency is the row worth locating.
TF_FREQ_MARKS: list[float] = [ASSR_FREQ]
MARK_STIMULUS_ONSETS_ON_TF = True

# ── Stimulus-locked epoch ─────────────────────────────────────────────
# The paradigm window, taken from src.definitions.constants.AssrEpoch so this notebook
# and the headless scripts cut the SAME epoch. iva_quality.onset_window caps the
# post-onset span by the shortest inter-onset gap, so an epoch never reaches the next
# stimulus.
EPOCH_PRE_S = AssrEpoch.PRE_ONSET_S
EPOCH_POST_S = AssrEpoch.POST_ONSET_S
MIN_ONSETS_FOR_EPOCH_AVERAGE = 5

# ── IVA settings ──────────────────────────────────────────────
# N_COMPONENTS_PCA reduces the *channel* axis, so it must be <= n_channels. The band
# run is exactly the case where the right value may differ from the broadband one: a
# 21-bin tensor is a simpler thing to summarise in K channel directions, which is why
# the job keeps an n_pca sweep. 5 is what run_condition_comparison_gamma.pbs runs.
N_COMPONENTS_PCA = 5
IVA_OPT_APPROACH = "newton"  # 'gradient', 'newton', or 'quasi'
IVA_MAX_ITER = 64  # the job uses 1024; 64 keeps the notebook interactive
IVA_W_DIFF_STOP = 1e-6
IVA_RANDOM_STATE = 42  # seeds per-recording PCA + W_init

# Run the broadband decomposition too, so Step 5 can put the two side by side. It is
# the more expensive of the pair (the sample axis is ~2.4x longer); set False once the
# band run is the only one being tuned.
RUN_BROADBAND_COMPARISON = True

# ── Anchor ────────────────────────────────────────────────────
ANCHOR_COORDINATE_SYSTEM = CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
# False accepts the intersection when a montage is missing some anchor electrodes
# (the --lenient_mask of the CLI scripts).
ANCHOR_MASK_STRICT = True

# ── Wavelet cache directories ─────────────────────────────────
# Both are BROADBAND directories, and deliberately so: there is no per-band cache
# anywhere in this project. The band is a slice of the 1-50 Hz tensor, taken in
# memory after loading, which is what lets a band run and a broadband run be compared
# bin for bin — they came from the same array.
WAVELET_DIR: Path = (
    resolve_wavelet_dir(None, EXPERIMENT_NAME) / SpectrumTypeVariants.BROADBAND.value
)
WAVELET_SUBSET_CACHE_DIR: Path = (
    resolve_notebook_wavelet_cache_dir(EXPERIMENT_NAME)
    / SpectrumTypeVariants.BROADBAND.value
)
REUSE_WAVELET_SUBSET_CACHE = True

# ── Plots directory ───────────────────────────────────────────
# Canonical notebook layout: plots/<experiment>/<broadband|bands>/<analysis_type>/.
# A band run goes under 'bands/' with the band as the first filename token, so it
# never lands on top of the broadband products — the same rule the CLI scripts follow.
SPECTRUM_DIR = (
    SpectrumTypeVariants.BROADBAND.value
    if BAND_NAME is None
    else SpectrumTypeVariants.BANDS.value
)
PLOT_PREFIX = "" if BAND_NAME is None else f"{BAND_NAME}_"
PLOTS_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR
    / "06-iva-condition-comparison"
    / "plots"
    / EXPERIMENT_NAME.value
    / SPECTRUM_DIR
    / "iva_channel_joined"
    / f"pca_{N_COMPONENTS_PCA}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

_bounds = resolve_band_range(BAND)
print(f"Experiment / group  : {EXPERIMENT_NAME.value} — {CONDITION.value}")
print(f"Conditions pooled   : {[c.value for c in CONDITIONS_TO_POOL]} (subject axis)")
print(
    f"Band restriction    : {BAND_NAME or 'none (broadband)'}"
    + ("" if _bounds is None else f"  = {_bounds[0]:g}–{_bounds[1]:g} Hz")
)
print(f"Stimulation freq    : {ASSR_FREQ:g} Hz")
print(f"Wavelet source cache: {WAVELET_DIR}")
print(
    f"Wavelet subset cache: {WAVELET_SUBSET_CACHE_DIR}  (reuse: {REUSE_WAVELET_SUBSET_CACHE})"
)
print(f"Frequencies         : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)")
print(f"Per-recording PCA   : {N_COMPONENTS_PCA}  (over channels)")
print(f"IVA optimisation    : {IVA_OPT_APPROACH}  (max_iter={IVA_MAX_ITER})")
print(f"Plots directory     : {PLOTS_DIR}  (saving: {SAVE_PLOTS})")
print(f"Figure prefix       : {PLOT_PREFIX!r}")

## Data Loading — both conditions on one subject axis

One call does the whole join: load each condition's **broadband** wavelet power from its
existing cache, keep the participants present in both, and stack them on the subject
axis. Identical to the broadband notebooks, because the band is not a loading concern —
it is applied to the array this returns.

**First run vs. every run after.** The source-of-truth wavelet tensor is decompressed in
full before it is trimmed to the requested channel/time extent (the stimulus alignment
is defined over the whole cohort, so that cache cannot be written per subset) — tens of
GB per condition for ASSR. The two conditions are loaded and trimmed **one at a time**,
so the peak is one condition's cache, not both at once.

The trimmed result is written to `WAVELET_SUBSET_CACHE_DIR`, and later runs at the same
extent read it back without opening the big cache. That cache is keyed by condition,
frequency grid and extent — **not** by the notebook that wrote it — so the entries this
notebook fills in are the ones the 03/04/05 workflows read, and vice versa.

In [ ]:
pooled, condition_analyzers = load_joined_condition_wavelets(
    MUSIC_TYPE,
    EXCLUSION_CATEGORIES,
    FREQS,
    wavelet_dir=WAVELET_DIR,
    experiment_name=EXPERIMENT_NAME,
    representation=REPRESENTATION,
    conditions=CONDITIONS_TO_POOL,
    n_channels=N_CHANNELS_SUBSET,
    n_times=N_TIMES_SUBSET,
    reuse_wavelets=REUSE_WAVELETS,
    subset_cache_dir=WAVELET_SUBSET_CACHE_DIR,
    reuse_subset_cache=REUSE_WAVELET_SUBSET_CACHE,
)

print(f"Pooled dataset : {pooled.data.label}")
print(f"Participants   : {pooled.n_pairs}  ->  {pooled.n_subjects} subjects")
print(
    f"Shape          : {pooled.data.data.shape}  (subjects × channels × freqs × times)"
)

# Restrict the cohort by *participant*, so both of a participant's recordings are
# kept and the pooled axis stays balanced.
if N_PAIRS_SUBSET is not None:
    keep = sorted(set(pooled.participants))[:N_PAIRS_SUBSET]
    pooled = pooled.select_participants(keep)
    print(f"\nUsing first {len(keep)} participant(s): {keep}")
    print(f"Shape          : {pooled.data.data.shape}")

print(f"\nSubject axis   : {list(pooled.subject_labels)}")

## Dataset Selection

There is only one dataset — the pooled one — so this cell unpacks it into the names the
steps below use and records the per-recording bookkeeping the pooled layout needs: which
participant and which condition each row is. `broadband_data` keeps the **whole** 1–50 Hz
tensor; the band is taken from it in Step 1 and the two coexist, which is what makes the
band-vs-broadband comparison a comparison of decompositions rather than of datasets.

In [ ]:
LABEL = f"{ConditionVariants.JOINED.value}_{MUSIC_TYPE.value}"

broadband_ad = pooled.data
broadband_data = broadband_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = broadband_ad.sfreq

n_subjects, n_channels, n_freqs_full, n_times = broadband_data.shape
time = np.arange(n_times) / sfreq

# Per-recording bookkeeping: which participant and which condition each row is.
subject_participants = list(pooled.participants)
subject_conditions = [c.value for c in pooled.subject_conditions]
subject_labels = list(pooled.subject_labels)
condition_masks = {c: pooled.condition_mask(c) for c in pooled.conditions}
CONDITION_ROWS = [c.value for c in pooled.conditions]

# NOT pooled.data.label: that is a *data* label and the wavelet transform appends the
# representation to it ("Joined_ASSR (wavelet power 1-50 Hz)"). Using it would put
# spaces, parentheses and a frequency range into every figure filename and would
# disagree with the component store, which names the condition from the enum.
print(f"Dataset    : {LABEL}   (data label: {broadband_ad.label})")
print(f"Shape      : {broadband_data.shape}  (recordings × channels × freqs × times)")
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs_full} steps)")
for condition, mask in condition_masks.items():
    print(
        f"  {condition.value:<12}: {int(mask.sum())} recording(s) at rows "
        f"{np.flatnonzero(mask).tolist()}"
    )

# ── Stimulus onset markers ────────────────────────────────────
# Both conditions share the aligned time base, so one set of onsets describes every
# recording; they can differ by a sample or two of rounding, so the first pooled
# condition's onsets are the reference. ``None`` for experiments without stimulus
# annotations (e.g. PSILO_MUSIC) -> no markers.
_onset_samples = pooled.condition_onsets(pooled.conditions[0])
if _onset_samples is None:
    stimulus_onset_times = np.array([])
    onset_samples_in = np.array([], dtype=int)
else:
    onset_samples_in = _onset_samples[_onset_samples < n_times].astype(int)
    stimulus_onset_times = onset_samples_in / sfreq
print(f"Stimulus onsets in window : {len(stimulus_onset_times)}")

# Topomap layout: the analyser's Info restricted to the channel subset. Both conditions
# were preprocessed onto the same montage, so either analyser will do.
iva_info = topo_info_subset(condition_analyzers[pooled.conditions[0]].info, n_channels)
iva_ch_names = list(iva_info["ch_names"])
print(
    f"Topomap channels          : {len(iva_ch_names)} "
    f"(first={iva_ch_names[0]}, last={iva_ch_names[-1]})"
)

---
## Step 1 — Slice the frequency axis to the band

`slice_to_band` keeps the bins of the 1–50 Hz grid inside the window and returns the
sliced tensor **together with its own frequency axis**. Both halves matter: after this
cell there are **two frequency grids in play**, and they do not share bin indices.

| Grid | Length | Bin of 40 Hz |
|------|--------|--------------|
| `FREQS` (broadband cache / store) | 50 | 39 |
| `band_freqs` (band store) | 21 | 10 |

That table is the whole hazard of band-restricted work, and the cell prints it for the
band actually configured. Bin 10 is 40 Hz on the band grid and **11 Hz** on the
broadband one. Any index resolved on one grid and used on the other silently reads the
wrong part of the spectrum — and does so *quietly*, because 11 Hz is a perfectly
plausible-looking alpha map. Resolve a frequency on the grid you are about to index,
every time, with `at.frequency_selection`; Step 4 checks that this held.

It is applied **before** the decomposition, which is the point:
`decompose_channel_iva` flattens frequency × time into one sample axis, so every bin
left here is a bin the PCA's variance ordering and `iva_g`'s dependence cost are
computed from. Slicing afterwards would restrict the figures and change nothing about
what was decomposed.

In [ ]:
band_data, band_freqs = slice_to_band(broadband_data, FREQS, BAND)
n_freqs_band = band_data.shape[2]

# The 40 Hz row on EACH grid, each resolved on its own. Never carry one of these
# indices to the other grid.
bin_band = int(at.frequency_selection(band_freqs, ASSR_FREQ)[0])
bin_full = int(at.frequency_selection(FREQS, ASSR_FREQ)[0])

print(
    f"Band            : {BAND_NAME}  ({resolve_band_range(BAND)[0]:g}–"
    f"{resolve_band_range(BAND)[1]:g} Hz)"
)
print(
    f"Broadband tensor: {broadband_data.shape}  "
    f"({n_freqs_full} bins, {FREQS[0]:g}–{FREQS[-1]:g} Hz)"
)
print(
    f"Band tensor     : {band_data.shape}  "
    f"({n_freqs_band} bins, {band_freqs[0]:g}–{band_freqs[-1]:g} Hz)"
)
print(
    f"Sample axis F*T : {n_freqs_band * n_times:,} vs {n_freqs_full * n_times:,} "
    f"broadband ({100 * n_freqs_band / n_freqs_full:.0f}%)"
)

print(
    f"\n{ASSR_FREQ:g} Hz is bin {bin_band} of the BAND grid, "
    f"bin {bin_full} of the BROADBAND grid."
)
print(f"  band grid bin {bin_band}      -> {band_freqs[bin_band]:g} Hz")
print(
    f"  broadband grid bin {bin_band} -> {FREQS[bin_band]:g} Hz   <- what a "
    "carried-over index would read"
)

# The band must actually contain the frequency the analysis is about; a window that
# misses it makes every figure below meaningless rather than merely narrow.
assert band_freqs[0] <= ASSR_FREQ <= band_freqs[-1], (
    f"BAND {BAND_NAME} does not contain {ASSR_FREQ:g} Hz; nothing below is about "
    "the ASSR."
)
print(f"\nOK: the band contains {ASSR_FREQ:g} Hz.")

---
## Step 2 — Decompose, band-restricted and broadband

One call each to
[`decompose_channel_iva`](../../src/analysis/iva_condition_comparison.py) — the same
function the CLI script calls, so what is tuned here is what the job runs. Per
recording: z-score along time, reshape to `(C, F·T)`, reduce the **channel** axis with
its own PCA to `N_COMPONENTS_PCA`, then `iva_g` across recordings, then recover the
spectro-temporal sources `(S, K, F, T)` and the forward channel patterns `(S, K, C)`.

Each of a participant's two recordings is its own dataset as far as IVA is concerned,
and IVA's permutation ambiguity is **shared across datasets** — component *k* means the
same thing in a Placebo recording and in a Psilocybin one, with no post-hoc matching.

The two runs are **independent decompositions of the same recordings**. Component 1 of
the band run is not component 1 of the broadband run, and the comparison in Step 5 is
therefore a comparison of what each set of components *is*, not of matched pairs.

This is the slow cell. `IVA_MAX_ITER` is 64 here against the job's 1024, so expect the
convergence warning on the full cohort — that is the tuning signal, not a failure.

In [ ]:
band_sources, band_patterns = decompose_channel_iva(
    band_data,
    label=f"{LABEL} [{BAND_NAME}]",
    n_pca=N_COMPONENTS_PCA,
    random_state=IVA_RANDOM_STATE,
    iva_opt_approach=IVA_OPT_APPROACH,
    iva_max_iter=IVA_MAX_ITER,
    iva_w_diff_stop=IVA_W_DIFF_STOP,
)
print(f"Band sources  : {band_sources.shape}  (recordings, K, F, T)")
print(f"Band patterns : {band_patterns.shape}  (recordings, K, C)")

if RUN_BROADBAND_COMPARISON:
    full_sources, full_patterns = decompose_channel_iva(
        broadband_data,
        label=f"{LABEL} [broadband]",
        n_pca=N_COMPONENTS_PCA,
        random_state=IVA_RANDOM_STATE,
        iva_opt_approach=IVA_OPT_APPROACH,
        iva_max_iter=IVA_MAX_ITER,
        iva_w_diff_stop=IVA_W_DIFF_STOP,
    )
    print(f"\nBroadband sources  : {full_sources.shape}")
    print(f"Broadband patterns : {full_patterns.shape}")
else:
    full_sources = full_patterns = None
    print("\nBroadband comparison skipped (RUN_BROADBAND_COMPARISON = False).")

COMP_INDICES = (
    list(range(N_COMPONENTS_PCA)) if COMPONENTS_TO_PLOT is None else COMPONENTS_TO_PLOT
)
print(f"\nComponents in the comparison figures: {[k + 1 for k in COMP_INDICES]}")

---
## Step 3 — Anchor every component's sign to the ASSR electrodes

`iva_g` fixes each component only up to a **per-recording** sign, so recording *i*'s copy
of component *k* may be the negation of recording *j*'s. Averaging unresolved maps drives
the group mean toward zero — with *S* recordings, to roughly `1/sqrt(S)` of the aligned
one — and on a *pooled* dataset it is worse than a lost mean: the flips fall arbitrarily
across the two condition blocks, so an **unresolved sign manufactures a condition
difference out of nothing**.

The anchor is `corr(topography, 0/1 ASSR mask)` across the whole channel axis. Since
`cov(pattern, mask) = f(1-f)(mean_inside - mean_outside)` for a binary mask, that asks
whether the pattern is more positive over the ASSR area **than over the rest of the
head** — so a pattern riding on a global offset cannot flip it. It reads the topography
alone and never the tested response, so it stays symmetric in the conditions and cannot
manufacture a contrast.

It is applied per **(recording, component)** — each component carries its own independent
sign — and to the sources *and* the patterns together, so a component's map and its
topography never disagree about which way is up.

**The anchor is band-independent**, and it is worth seeing why: the mask is a list of
electrode names, the correlation is over channels, and neither knows the frequency axis
exists. The band and broadband runs are anchored by the identical operator.

In [ ]:
# The 0/1 electrode selection, on this run's own channel axis.
assr_anchor_mask = assr_electrode_mask(
    iva_ch_names, ANCHOR_COORDINATE_SYSTEM, strict=ANCHOR_MASK_STRICT
)
print(
    f"Anchor electrodes present: {int(assr_anchor_mask.sum())} of {n_channels} "
    f"channel(s) in this subset"
)


def anchor_to_assr(sources, patterns, tag):
    """Flip every (recording, component) so positive = more power over the ASSR area."""
    flip, strength = at.polarity_flip(patterns, assr_anchor_mask)
    n_weak, n_total = at.polarity_weak_count(strength)
    print(
        f"\n[{tag}] {int((flip < 0).sum())}/{flip.size} (recording, component) "
        f"pair(s) flipped; {n_weak}/{n_total} decided on |corr| < "
        f"{at.POLARITY_CORR_FLOOR} (median |corr| {np.median(strength):.3f})"
    )
    if n_weak:
        print(
            "  NOTE: a weakly anchored recording CANCELS signal in a group mean "
            "rather than merely widening it."
        )
    return (
        apply_component_signs(sources, flip),
        apply_component_signs(patterns, flip),
    )


band_sources, band_patterns = anchor_to_assr(band_sources, band_patterns, BAND_NAME)
if RUN_BROADBAND_COMPARISON:
    full_sources, full_patterns = anchor_to_assr(
        full_sources, full_patterns, "broadband"
    )

# The note every figure below prints, naming the anchor actually in force.
ALIGNMENT_NOTE = "corr(topography, ASSR electrode mask), per (recording, component)"

---
## Step 4 — The fixed 40 Hz reference must not move

**The check this notebook exists for.** The reference every component is judged against
in the stage-06 statistics is the **binary ASSR-electrode mask** applied to the wavelet
power at 40 Hz: no PCA, no IVA, no learned weights — an equal-weight average over a fixed
list of electrodes, read out of the broadband cache. Nothing about a decomposition enters
it, so **it is the same number in a band run and in a broadband one**, and it has been
the same number since the 40 Hz row was first read. If a band run draws a different
reference, no hypothesis about the band explains it: an index was resolved on one
frequency grid and used against the other, and in that case the *components* are being
read out of the wrong frequencies too — the reference is merely where it is visible,
because it is the one row whose correct value is known in advance.

This cell computes the reference two ways — off the broadband tensor at its own 40 Hz
bin, and off the band tensor at *its* own 40 Hz bin — and asserts they are the same
array. They are slices of one tensor at one frequency, so anything but an exact match is
a bug in how the bin was resolved.

The deliberate mistake is shown beside it: the **band** grid's bin index used against the
**broadband** grid. That is the failure mode, and on this grid it reads 11 Hz — alpha —
which is exactly why it survives a glance at the figure.

In the scripts the same care appears as two selections resolved separately,
`store_selections` on `results.freqs` and `cache_selections` on the cache header's grid
(`scripts/run_assr_snr_grid.py`), because there the store may hold only the band's bins
while the streamed cache is always broadband.

In [ ]:
# Equal-weight average over the anchor electrodes — the reference row, as
# scripts/run_assr_snr_grid.py builds it ("ASSR-mask (full)").
binary = at.binary_filter_weights(assr_anchor_mask, normalize=True)

# Resolved on the BROADBAND grid, applied to the broadband tensor.
ref_from_full = np.stack(
    [
        at.project_channels(binary[None, :], rec[:, bin_full, :])[0]
        for rec in broadband_data
    ]
)  # (S, T)

# Resolved on the BAND grid, applied to the band tensor. Different index, same hertz.
ref_from_band = np.stack(
    [at.project_channels(binary[None, :], rec[:, bin_band, :])[0] for rec in band_data]
)  # (S, T)

print(
    f"Reference from broadband tensor, bin {bin_full} ({FREQS[bin_full]:g} Hz)  : "
    f"{ref_from_full.shape}"
)
print(
    f"Reference from band tensor,      bin {bin_band} ({band_freqs[bin_band]:g} Hz)  : "
    f"{ref_from_band.shape}"
)
print(f"Max |difference| : {np.abs(ref_from_full - ref_from_band).max():.3e}")
assert np.array_equal(ref_from_full, ref_from_band), (
    "The fixed ASSR-electrode reference differs between the band and broadband "
    f"tensors at {ASSR_FREQ:g} Hz. Both are the SAME slice of the SAME cache, so one "
    "of the two bin indices was resolved on the wrong grid."
)
print(
    f"\nOK: the {ASSR_FREQ:g} Hz reference is identical whichever grid it is "
    "resolved on."
)

# And the mistake, for contrast: the band grid's index used on the broadband grid.
ref_wrong = np.stack(
    [
        at.project_channels(binary[None, :], rec[:, bin_band, :])[0]
        for rec in broadband_data
    ]
)  # (S, T) — but at FREQS[bin_band] Hz, not 40 Hz
_corr = np.corrcoef(ref_from_full.ravel(), ref_wrong.ravel())[0, 1]
print(
    f"\nThe deliberate mistake — band bin {bin_band} read on the broadband grid — "
    f"lands on {FREQS[bin_band]:g} Hz.\n"
    f"  corr(correct, wrong) = {_corr:+.3f}, ratio of means = "
    f"{ref_wrong.mean() / ref_from_full.mean():.2f}x\n"
    "  It is a perfectly well-formed time course, which is why it is not caught by "
    "eye."
)

# Same story one level up: the reference TOPOGRAPHY at 40 Hz, mean over time and
# recordings. This is the "full ASSR map" a band run must reproduce exactly.
ref_topo_full = broadband_data[:, :, bin_full, :].mean(axis=(0, 2))  # (C,)
ref_topo_band = band_data[:, :, bin_band, :].mean(axis=(0, 2))  # (C,)
assert np.allclose(ref_topo_full, ref_topo_band)
print(
    f"\nOK: the {ASSR_FREQ:g} Hz reference topography is identical too "
    f"({ref_topo_full.shape[0]} channels)."
)

---
## Step 5 — Condition-mean comparison, band vs broadband

The overview grid: **rows = conditions, columns = components**, so every component's
Placebo and Psilocybin means sit one above the other, with a difference row
(Psilocybin − Placebo) because a small difference between two similar maps is far easier
to see drawn than inferred.

Scaling rules, and why: each column has **one symmetric limit shared by the condition
rows** (a comparison drawn on two independent scales is not a comparison), and limits are
**not** shared across columns (IVA fixes each component's scale independently, so a common
limit would render the weaker components flat). Each column's `|max|` is in its title,
which is also why the grid carries no single colourbar.

**What to look for.** The band run's TF maps span 30–50 Hz only, so the 40 Hz line is a
fifth of the way down a 21-bin axis rather than a sliver at the top of a 50-bin one. The
question the band is meant to answer is whether *more* of the `N_COMPONENTS_PCA`
components now carry a 40 Hz response at all — in the broadband run the objective is free
to spend them on drift and alpha, which are far more reproducible across recordings than
a driven response. Read the topographies against the ASSR electrode mask, not against
each other.

Figures go under `bands/` with the band as the first filename token, so they sit beside
the broadband ones rather than replacing them — the same layout rule the CLI scripts
follow.

In [ ]:
tf_time_marks = stimulus_onset_times if MARK_STIMULUS_ONSETS_ON_TF else None


def draw_condition_means(sources, patterns, freqs, tag, prefix):
    """The two condition-mean grids for one decomposition."""
    fig_topo = plot_condition_mean_topomaps(
        patterns,
        subject_participants,
        subject_conditions,
        CONDITION_ROWS,
        iva_info,
        n_channels,
        COMP_INDICES,
        label=f"{LABEL} [{tag}]",
        alignment_note=ALIGNMENT_NOTE,
        save_path=(
            (PLOTS_DIR / f"{prefix}condition_mean_topomaps.png") if SAVE_PLOTS else None
        ),
    )
    plt.show()
    plt.close("all")

    fig_tf = plot_condition_mean_tf_maps(
        sources,
        subject_participants,
        subject_conditions,
        CONDITION_ROWS,
        freqs,
        time,
        COMP_INDICES,
        label=f"{LABEL} [{tag}]",
        time_marks=tf_time_marks,
        freq_marks=TF_FREQ_MARKS,
        alignment_note=ALIGNMENT_NOTE,
        save_path=(
            (PLOTS_DIR / f"{prefix}condition_mean_tf_maps.png") if SAVE_PLOTS else None
        ),
    )
    plt.show()
    plt.close("all")
    return fig_topo, fig_tf


draw_condition_means(band_sources, band_patterns, band_freqs, BAND_NAME, PLOT_PREFIX)

if RUN_BROADBAND_COMPARISON:
    # Written next to the band figures with a 'broadband_' prefix rather than into the
    # broadband directory: this is the notebook's own side-by-side, not a substitute
    # for the broadband run's products.
    draw_condition_means(full_sources, full_patterns, FREQS, "broadband", "broadband_")

if SAVE_PLOTS:
    print(f"Saved condition-mean figures to {PLOTS_DIR}")

---
## Step 6 — Per participant

One figure per component, **Placebo on the first row and Psilocybin on the second**, one
column per participant, columns ordered by participant ID — so a participant's two
recordings sit directly one above the other and the same participant holds the same
column in every figure.

That vertical pairing is the point. The design is within-participant (the pooling keeps
only participants recorded under both conditions), so this answers what the means cannot:
is a difference in the condition means shared across the group, or carried by one or two
people? Here **every panel of a figure shares one colour limit**, across participants and
conditions, so the whole grid is comparable and the single colourbar means something.

On the band run this is also the clearest read on the sign anchor: a participant whose
topography is the negative of their neighbours' got a weak anchor in Step 3, and a weak
anchor cancels signal in the condition mean above rather than merely widening it.

In [ ]:
topo_paths = plot_participant_condition_topomaps(
    band_patterns,
    subject_participants,
    subject_conditions,
    CONDITION_ROWS,
    iva_info,
    n_channels,
    COMP_INDICES,
    label=f"{LABEL} [{BAND_NAME}]",
    root_dir=PLOTS_DIR / "participants",
    prefix=PLOT_PREFIX,
    alignment_note=ALIGNMENT_NOTE,
)
tf_paths = plot_participant_condition_tf_maps(
    band_sources,
    subject_participants,
    subject_conditions,
    CONDITION_ROWS,
    band_freqs,
    time,
    COMP_INDICES,
    label=f"{LABEL} [{BAND_NAME}]",
    root_dir=PLOTS_DIR / "participants",
    prefix=PLOT_PREFIX,
    time_marks=tf_time_marks,
    freq_marks=TF_FREQ_MARKS,
    alignment_note=ALIGNMENT_NOTE,
)
print(
    f"Wrote {len(topo_paths)} topography and {len(tf_paths)} TF figure(s) to "
    f"{PLOTS_DIR / 'participants'}"
)

for path in topo_paths + tf_paths:
    display(Image(filename=str(path)))

---
## Step 7 — Stimulus-locked comparison (averaged over onsets)

Steps 5 and 6 draw each component's TF map over the **whole** recording. That answers
"what does this component do", but not "what does it do *to a stimulus*": the ASSR is a
train of short stimuli, and a response locked to their onsets is smeared across a map
that spans every stimulus at once.

So: cut a fixed epoch around every onset and average it, per recording and per component.
A `(F, T)` whole-recording map becomes an `(F, W)` onset-averaged one, and the condition
contrast becomes a contrast of stimulus responses. The epoch is the paradigm window from
[`AssrEpoch`](../../src/definitions/constants.py) via `iva_quality.onset_window`, the
same call the headless scripts make.

Each frequency is then referenced to its own pre-onset mean. The z-scoring put the
pre-onset interval near 0 over the **whole** recording but not within any single epoch —
the local level still drifts — so this is what turns the map into a *change* rather than
a level. Mean only, no divisor: 1/f is already gone, and at the bottom of the map a
100 ms baseline spans under one cycle, so its SD is mostly wavelet phase and dividing by
it would manufacture texture there. On a 30–50 Hz band that last caveat does not bite —
every bin of the band has many cycles in the baseline — but the subtraction is kept
identical to the broadband path so the two remain comparable.

**This is the figure the band is for.** A driven 40 Hz response is a horizontal stripe
starting at t = 0 and ending at the stimulus offset (the two marked lines). With the
notebook's short `N_TIMES_SUBSET` only a handful of epochs fit the window, so treat it as
a shape check and read the real thing off the job's output.

In [ ]:
# The paradigm window, capped by the shortest gap so epochs never overlap.
if onset_samples_in.size:
    EPOCH_PRE, EPOCH_POST = iva_quality.onset_window(onset_samples_in, n_times, sfreq)
    n_fitting = int(
        (
            (onset_samples_in - EPOCH_PRE >= 0)
            & (onset_samples_in + EPOCH_POST <= n_times)
        ).sum()
    )
else:
    EPOCH_PRE = EPOCH_POST = 0
    n_fitting = 0

run_epoch_comparison = n_fitting >= MIN_ONSETS_FOR_EPOCH_AVERAGE
if not run_epoch_comparison:
    print(
        f"Stimulus-locked comparison skipped: {n_fitting} epoch(s) fit the "
        f"{n_times}-sample window, need {MIN_ONSETS_FOR_EPOCH_AVERAGE}. "
        "Raise N_TIMES_SUBSET (or use an experiment with stimulus annotations)."
    )
else:
    onset_tf, n_used = iva_quality.epoch_average(
        band_sources, onset_samples_in, EPOCH_PRE, EPOCH_POST
    )
    epoch_times = np.arange(-EPOCH_PRE, EPOCH_POST) / sfreq  # t=0 at the onset
    onset_tf = iva_quality.subtract_epoch_baseline(onset_tf, epoch_times < 0.0)
    # Onset (dashed) and stimulus offset (dotted), clamped so the offset line is never
    # drawn outside the epoch.
    EPOCH_MARKS = [0.0, min(AssrEpoch.STIMULUS_DURATION_S, float(epoch_times[-1]))]

    print(f"Onset-averaged TF maps : {onset_tf.shape}  (S, K, F, W)")
    print(f"Epochs averaged        : {n_used} of {len(onset_samples_in)} onset(s)")
    print(
        f"Epoch window           : {EPOCH_PRE + EPOCH_POST} samples "
        f"({EPOCH_PRE} pre, {EPOCH_POST} post) = "
        f"[{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s"
    )

    fig_tf_onset = plot_condition_mean_tf_maps(
        onset_tf,
        subject_participants,
        subject_conditions,
        CONDITION_ROWS,
        band_freqs,
        epoch_times,
        COMP_INDICES,
        label=f"{LABEL} [{BAND_NAME}]",
        freq_marks=TF_FREQ_MARKS,
        epoch_marks=EPOCH_MARKS,
        alignment_note=ALIGNMENT_NOTE,
        save_path=(
            (PLOTS_DIR / f"{PLOT_PREFIX}condition_mean_tf_maps_onset.png")
            if SAVE_PLOTS
            else None
        ),
    )
    plt.show()
    plt.close("all")

---
## Step 8 — Scale it up: the job this notebook stands in for

Nothing here writes a component store — the notebook runs on a channel/time subset, and a
store written from it would be indistinguishable by filename from the real one. When the
band, `N_COMPONENTS_PCA` and the anchor look right, run the same pipeline on the whole
cohort through
[`run_condition_comparison_gamma.pbs`](../../jobs/metacentrum/06-iva-condition-comparison/run_condition_comparison_gamma.pbs),
which is the cell below plus the Singularity and scratch boilerplate.

The job writes, side by side with the broadband products and never on top of them:

```
data/processed/assr/iva_results/Joined/
    iva_channel_joined__ASSR__30-50hz__pca<N>.npz
plots/06-iva-condition-comparison/Joined_ASSR/bands/iva_channel_joined/pca_<N>/
    30-50hz_*.png
```

**Resources.** A band run shrinks the IVA sample axis to ~40% but **not** the load-time
peak: the broadband cache is read in full and sliced after, so the job's `mem` must stay
as it is unless `--n_times` is also set.

**Then the statistics.** `--band 30-50hz` on
[`run_snr_grid_joined_gamma.pbs`](../../jobs/metacentrum/06-iva-condition-comparison/run_snr_grid_joined_gamma.pbs)
selects which *store* to read and nothing else: the wavelet cache those scripts stream is
the broadband one either way, and their frequency selections are resolved on the cache's
own grid. Which is Step 4's invariant, stated as a job: the ASSR-electrode reference the
band components are judged against is the same reference the broadband components were
judged against.

In [ ]:
_low, _high = resolve_band_range(BAND)
print(
    "python scripts/run_iva_condition_comparison.py \\\n"
    "    --experiment assr \\\n"
    "    --conditions Placebo Psilocybin \\\n"
    f"    --band_range {_low:g} {_high:g} \\\n"
    f"    --n_pca {N_COMPONENTS_PCA} \\\n"
    f"    --random_state {IVA_RANDOM_STATE} \\\n"
    f"    --iva_opt_approach {IVA_OPT_APPROACH} \\\n"
    "    --iva_max_iter 1024 \\\n"
    "    --reuse_wavelets \\\n"
    "    --store_components \\\n"
    "    --n_jobs -1"
)
print(
    f"\n-> store  data/processed/assr/iva_results/{ConditionVariants.JOINED.value}/"
    f"iva_channel_joined__{MUSIC_TYPE.value}__{BAND_NAME}__pca{N_COMPONENTS_PCA}.npz"
)
print(
    f"-> plots  plots/06-iva-condition-comparison/{LABEL}/"
    f"{SpectrumTypeVariants.BANDS.value}/iva_channel_joined/"
    f"pca_{N_COMPONENTS_PCA}/{PLOT_PREFIX}*.png"
)